In [ ]:
#Veri oluşturalım
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 100
df = pd.DataFrame({
    "Calisan_Sayisi": np.random.randint(10, 25, n),
    "Makine_Suresi": np.random.uniform(6, 10, n),
    "Enerji_Tuketimi": np.random.uniform(350, 500, n),
    "Hammadde_Kalite": np.random.randint(1, 6, n)
})

df["Uretim_Miktari"] = (
    20 * df["Calisan_Sayisi"] +
    30 * df["Makine_Suresi"] +
    5 * df["Hammadde_Kalite"] +
    np.random.normal(0, 50, n)
)

df.head() #Verinin ilk 5 satırını görüntüleyelim

,Calisan_Sayisi,Makine_Suresi,Enerji_Tuketimi,Hammadde_Kalite,Uretim_Miktari
0,16,9.757996,371.734231,1,590.277228
1,13,9.579309,423.417914,3,617.219095
2,22,8.391600,497.847568,2,807.403605
3,24,9.687497,386.308291,1,750.704577
4,20,6.353970,450.820332,2,584.399328


In [ ]:
df.isnull().sum() #Eksik değer var mı kontrol edelim.

In [ ]:
#Gürültü Kontrolü (Basit görsel inceleme)
plt.figure()
plt.plot(df["Uretim_Miktari"])
plt.title("Uretim_Miktari Zaman Benzeri Grafik (Gürültü İncelemesi)")
plt.show()

In [ ]:
#Aykırı değer kontrolü (Boxplot ile görsel inceleme)
plt.figure()
plt.boxplot(df["Uretim_Miktari"].dropna())
plt.title("Uretim_Miktari Boxplot (Aykırı Değer Kontrolü)")
plt.show()

# IQR yöntemi ile aykırı değer tespiti
Q1 = df["Uretim_Miktari"].quantile(0.25)
Q3 = df["Uretim_Miktari"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df["Uretim_Miktari"] < lower_bound) | (df["Uretim_Miktari"] > upper_bound)]

print("\nIQR Yöntemi ile Tespit Edilen Aykırı Değerler:\n")
print(outliers[["Uretim_Miktari"]])

In [ ]:
# Her satır = bir gözlem
# Her sütun = bir değişken
# Uretim_Miktari = bağımlı değişken
# Diğerleri = bağımsız değişken

In [2]:
df.corr() #Değişkenler arasındaki korelasyonu inceleyelim
# Üretim ile en güçlü ilişki hangi değişkende?
# Yüksek korelasyonlu iki bağımsız değişken var mı?

,Calisan_Sayisi,Makine_Suresi,Enerji_Tuketimi,Hammadde_Kalite,Uretim_Miktari
Calisan_Sayisi,1.000000,-0.069613,0.250909,-0.120273,0.784442
Makine_Suresi,-0.069613,1.000000,-0.035334,0.077733,0.292471
Enerji_Tuketimi,0.250909,-0.035334,1.000000,-0.066374,0.242210
Hammadde_Kalite,-0.120273,0.077733,-0.066374,1.000000,-0.025134
Uretim_Miktari,0.784442,0.292471,0.242210,-0.025134,1.000000


In [ ]:
# Yorum:
# En güçlü ilişki çalışan sayısı ile
# Hammadde kalitesi neredeyse etkisiz görünüyor
# Bağımsız değişkenler arasında aşırı yüksek korelasyon yok

In [3]:
#Yeni özellik üretelim (feature engineering):

df["Verimlilik"] = df["Uretim_Miktari"] / df["Enerji_Tuketimi"]
df["Calisan_x_Makine"] = df["Calisan_Sayisi"] * df["Makine_Suresi"]

In [4]:
df.head() #Yeni özelliklerle birlikte verinin ilk 5 satırını görüntüleyelim

,Calisan_Sayisi,Makine_Suresi,Enerji_Tuketimi,Hammadde_Kalite,Uretim_Miktari,Verimlilik,Calisan_x_Makine
0,16,9.757996,371.734231,1,590.277228,1.587901,156.127932
1,13,9.579309,423.417914,3,617.219095,1.457707,124.531022
2,22,8.391600,497.847568,2,807.403605,1.621789,184.615198
3,24,9.687497,386.308291,1,750.704577,1.943278,232.499927
4,20,6.353970,450.820332,2,584.399328,1.296302,127.079400


In [5]:
#Basit korelasyona göre seçim yapalım:

corr_matrix = df.corr()
corr_matrix["Uretim_Miktari"].sort_values(ascending=False)

Uretim_Miktari      1.000000
Verimlilik          0.845085
Calisan_x_Makine    0.826718
Calisan_Sayisi      0.784442
Makine_Suresi       0.292471
Enerji_Tuketimi     0.242210
Hammadde_Kalite    -0.025134
Name: Uretim_Miktari, dtype: float64

In [ ]:
# Yorum:
# Feature engineering gerçekten işe yaradı.
# Yeni değişkenler, orijinal değişkenlerden daha güçlü açıklayıcı hale geldi.

In [6]:
#Standardizasyon
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop("Uretim_Miktari", axis=1))

In [7]:
X_scaled[:5] #Standardize edilmiş ilk 5 satırı görüntüleyelim

array([[-0.30020118,  1.54095132, -1.24851076, -1.36954241,  0.64592693,
         0.51266195],
       [-1.00933781,  1.39101256, -0.07676026,  0.02085598,  0.15527982,
        -0.31073253],
       [ 1.11807209,  0.39438539,  1.61067723, -0.67434322,  0.77363466,
         1.25502118],
       [ 1.59082985,  1.48179456, -0.91809387, -1.36954241,  1.98519032,
         2.50286538],
       [ 0.64531434, -1.31542441,  0.54449572, -0.67434322, -0.45298421,
        -0.24432349]])

In [ ]:
#PCA Uygulaması
from sklearn.decomposition import PCA

pca = PCA()
X_pca = pca.fit_transform(X_scaled)

print(pca.explained_variance_ratio_)

# İlk 2 bileşen toplam varyansın % kaçını açıklıyor?
print("Toplam varyans açıklanma oranı (İlk 2 bileşen):", sum(pca.explained_variance_ratio_[:2]) * 100)
# Boyutu azaltabilir miyiz?

[0.42833782 0.22136474 0.17134363 0.15074996 0.02689802 0.00130583]
Toplam varyans açıklanma oranı (İlk 2 bileşen): 64.97025621329615


In [ ]:
# Yorum:
# İlk 2 bileşen verinin %65'ini açıklıyor.
# Bu şu demek:6 değişken yerine 2 bileşenle büyük ölçüde aynı bilgiyi taşıyabiliriz.

# Dikkat: PCA yorumlanabilirliği azaltır.
# Performans artışı için yorumlanabilirlikten vazgeçer miyiz?

In [ ]:
#Modelleme (Basit Regresyon)
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = df.drop("Uretim_Miktari", axis=1)
y = df["Uretim_Miktari"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R2:", r2_score(y_test, y_pred))

In [ ]:
# Yorum:?

In [ ]:
# Önce veriyi anla
# Korelasyon ile ilişkileri keşfet
# Feature engineering ile gücü artır
# Gereksiz değişkenleri tartış
# Ölçekleme yap
# Boyut azaltma alternatifini düşün
# Model kur ve değerlendir